In [3]:
# ============================================================
# PTQ — GPU + CPU Quantization for Fair Comparison with Pruning
# ResNet-50 / CIFAR-10
#
# Problem: PyTorch native PTQ (torch.ao) runs on CPU only.
#          Pruning results were measured on GPU.
#          Direct inference_ms comparison is invalid across devices.
#
# Solution: Two parallel tracks
#   Track A — CPU PTQ (torch.ao)
#       Best observer from your existing script (histogram/moving_avg).
#       Reports CPU inference_ms honestly.
#
#   Track B — GPU INT8 (bitsandbytes)
#       Replaces nn.Linear layers with bnb.nn.Linear8bitLt.
#       Runs on CUDA. Reports GPU inference_ms directly comparable
#       to pruning results.
#
#   Track C — GPU FP16 (torch.amp / half())
#       Simple .half() cast — no library needed.
#       2× memory reduction, GPU-native, fast baseline for comparison.
#       Not INT8 but widely used and honest.
#
# Output: __1__PTQ_v2.json
#
# Install: pip install bitsandbytes --break-system-packages
#          (skip Track B if unavailable — Tracks A and C still run)
# ============================================================

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
from torch.ao.quantization import QConfig, get_default_qconfig
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
import torch.quantization as tq
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────
BASELINE_PTH     = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_JSON    = "../__2__baseline_metrics.json"
OUTPUT_JSON      = "__1__PTQ_GPU.json"

GPU              = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CPU              = torch.device("cpu")
BATCH_SIZE       = 128
NUM_CLASSES      = 10
CALIB_SIZE       = 1024
CALIB_BATCHES    = 8
INFERENCE_RUNS   = 300
WARMUP_RUNS      = 20

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"GPU device : {GPU}")
print(f"CUDA avail : {torch.cuda.is_available()}")

# ── bitsandbytes availability check ───────────────────────────
try:
    import bitsandbytes as bnb
    BNB_AVAILABLE = True
    print("bitsandbytes : available")
except ImportError:
    BNB_AVAILABLE = False
    print("bitsandbytes : NOT installed — Track B skipped")
    print("  Install: pip install bitsandbytes --break-system-packages")

# ── Model factory ─────────────────────────────────────────────
def build_model(num_classes=NUM_CLASSES):
    m = models.resnet50(weights=None)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m

def load_fp32(device=CPU):
    m = build_model()
    m.load_state_dict(torch.load(BASELINE_PTH, map_location=device))
    return m.to(device).eval()

# ── Data ──────────────────────────────────────────────────────
def get_test_loader(device):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10('../data', False, download=True, transform=transform)
    return DataLoader(ds, BATCH_SIZE, shuffle=False, num_workers=0,
                      pin_memory=(device.type == 'cuda'))

def get_calib_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds  = torchvision.datasets.CIFAR10('../data', True, download=True, transform=transform)
    sub = Subset(ds, list(range(CALIB_SIZE)))
    return DataLoader(sub, BATCH_SIZE, shuffle=False, num_workers=0)

# ── Metric helpers ────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x = x.to(device)
        preds.extend(model(x).argmax(1).cpu().numpy())
        labels.extend(y.numpy())
    p, l = np.array(preds), np.array(labels)
    return dict(
        accuracy  = float(accuracy_score(l, p)),
        precision = float(precision_score(l, p, average='macro', zero_division=0)),
        recall    = float(recall_score(l, p, average='macro', zero_division=0)),
        f1        = float(f1_score(l, p, average='macro', zero_division=0)),
    )

def measure_gpu_ms(model, device, n=INFERENCE_RUNS, warmup=WARMUP_RUNS):
    """GPU timing using CUDA events — same method as pruning script."""
    if device.type != 'cuda':
        return None
    model.eval()
    
    # Match dummy dtype to model's parameter dtype (FP32 or FP16)
    param = next(model.parameters())
    dummy = torch.randn(1, 3, 32, 32, device=device, dtype=param.dtype)  # ← fix
    
    with torch.no_grad():
        for _ in range(warmup): model(dummy)
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(n): model(dummy)
        end.record()
        torch.cuda.synchronize()
    return start.elapsed_time(end) / n

def measure_cpu_ms(model, n=INFERENCE_RUNS):
    model = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(1, 3, 32, 32)
    with torch.no_grad():
        for _ in range(10): model(dummy)
        t0 = time.perf_counter()
        for _ in range(n): model(dummy)
    return (time.perf_counter() - t0) / n * 1000

def disk_mb(model, use_state_dict=True):
    with tempfile.NamedTemporaryFile(suffix='.pth', delete=False) as f:
        tmp = f.name
    try:
        if use_state_dict:
            torch.save(model.state_dict(), tmp)
        else:
            torch.save(model, tmp)  # needed for quantized models
        return round(os.path.getsize(tmp) / 1e6, 3)
    finally:
        os.remove(tmp)

@torch.no_grad()
def calibrate(model, loader, max_batches=CALIB_BATCHES):
    model.eval()
    for i, (x, _) in enumerate(loader):
        model(x)
        if i + 1 >= max_batches:
            break

# ══════════════════════════════════════════════════════════════
# TRACK A — CPU PTQ (torch.ao, best observer = histogram)
# This is your existing method, kept for completeness.
# inference_ms reported on CPU — NOT comparable to GPU pruning times.
# ══════════════════════════════════════════════════════════════
def run_track_a(baseline_acc):
    print("\n" + "="*60)
    print("TRACK A — CPU PTQ (torch.ao, histogram observer)")
    print("  Device: CPU | INT8 weights + activations")
    print("  NOTE: CPU timing — not directly comparable to GPU pruning")
    print("="*60)

    qconfig  = QConfig(
        activation=tq.HistogramObserver.with_args(
            dtype=torch.quint8, qscheme=torch.per_tensor_affine),
        weight=tq.PerChannelMinMaxObserver.with_args(
            dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
    )
    model    = load_fp32(CPU)
    example  = torch.randn(1, 3, 32, 32)
    prepared = prepare_fx(model, {"": qconfig}, example)
    calibrate(prepared, get_calib_loader())
    q_model  = convert_fx(prepared)
    q_model.eval()

    test_loader = get_test_loader(CPU)
    metrics     = evaluate(q_model, test_loader, CPU)
    cpu_ms      = measure_cpu_ms(q_model)
    size_mb     = disk_mb(q_model, use_state_dict=False)

    result = dict(
        track          = "A",
        method         = "CPU PTQ (torch.ao histogram)",
        device         = "cpu",
        dtype          = "INT8 (weights + activations)",
        **{k: round(v, 6) for k, v in metrics.items()},
        accuracy_drop  = round(baseline_acc - metrics['accuracy'], 6),
        size_mb        = size_mb,
        inference_cpu_ms = round(cpu_ms, 4),
        inference_gpu_ms = None,
        timing_note    = (
            "CPU timing only. PyTorch torch.ao quantization does not support "
            "CUDA execution. Do NOT compare this ms value directly with GPU "
            "pruning inference times — they measure different hardware."
        ),
    )
    print(f"  acc={metrics['accuracy']:.4f}  drop={result['accuracy_drop']:+.4f}  "
          f"size={size_mb:.2f} MB  cpu={cpu_ms:.2f} ms")
    return result, q_model

# ══════════════════════════════════════════════════════════════
# TRACK B — GPU INT8 via bitsandbytes
# Replaces nn.Linear with bnb.Linear8bitLt — runs on CUDA.
# inference_ms directly comparable to GPU pruning results.
# ══════════════════════════════════════════════════════════════
def _replace_linear_bnb(model):
    """Recursively replace nn.Linear with bnb.Linear8bitLt."""
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            new_layer = bnb.nn.Linear8bitLt(
                module.in_features,
                module.out_features,
                bias=(module.bias is not None),
                has_fp16_weights=False,
            )
            new_layer.weight = bnb.nn.Int8Params(
                module.weight.data.clone(),
                requires_grad=False,
                has_fp8_weights=False,
            )
            if module.bias is not None:
                new_layer.bias = nn.Parameter(module.bias.data.clone())
            setattr(model, name, new_layer)
        else:
            _replace_linear_bnb(module)
    return model

def run_track_b(baseline_acc):
    if not BNB_AVAILABLE:
        print("\nTRACK B — SKIPPED (bitsandbytes not installed)")
        return None, None

    if GPU.type != 'cuda':
        print("\nTRACK B — SKIPPED (no CUDA device)")
        return None, None

    print("\n" + "="*60)
    print("TRACK B — GPU INT8 (bitsandbytes Linear8bitLt)")
    print("  Device: CUDA | INT8 Linear weights, FP16 activations")
    print("  timing directly comparable to GPU pruning results")
    print("="*60)

    model = load_fp32(GPU)
    model = _replace_linear_bnb(model)
    model = model.to(GPU).eval()

    # Warm-up pass triggers INT8 quantization on first forward
    with torch.no_grad():
        model(torch.randn(1, 3, 32, 32, device=GPU))

    test_loader = get_test_loader(GPU)
    metrics     = evaluate(model, test_loader, GPU)
    gpu_ms      = measure_gpu_ms(model, GPU)
    size_mb     = disk_mb(model, use_state_dict=True)

    result = dict(
        track          = "B",
        method         = "GPU INT8 (bitsandbytes Linear8bitLt)",
        device         = str(GPU),
        dtype          = "INT8 (Linear weights) + FP32 (Conv activations)",
        **{k: round(v, 6) for k, v in metrics.items()},
        accuracy_drop  = round(baseline_acc - metrics['accuracy'], 6),
        size_mb        = size_mb,
        inference_cpu_ms = None,
        inference_gpu_ms = round(gpu_ms, 4) if gpu_ms else None,
        timing_note    = (
            "GPU timing via CUDA events (same method as pruning benchmarks). "
            "Only nn.Linear layers are INT8; Conv2d stays FP32. "
            "Primary compression effect is on the FC layer of ResNet-50."
        ),
    )
    print(f"  acc={metrics['accuracy']:.4f}  drop={result['accuracy_drop']:+.4f}  "
          f"size={size_mb:.2f} MB  gpu={gpu_ms:.2f} ms")
    return result, model

# ══════════════════════════════════════════════════════════════
# TRACK C — GPU FP16 (.half())
# No library needed. 2× memory reduction. Runs on CUDA.
# Not INT8 but a legitimate, widely-used precision reduction.
# inference_ms directly comparable to GPU pruning results.
# ══════════════════════════════════════════════════════════════
def run_track_c(baseline_acc):
    print("\n" + "="*60)
    print("TRACK C — GPU FP16 (torch .half())")
    print("  Device: CUDA | 16-bit floats, no calibration needed")
    print("  timing directly comparable to GPU pruning results")
    print("="*60)

    if GPU.type != 'cuda':
        print("  SKIPPED — no CUDA device")
        return None, None

    model       = load_fp32(GPU).half()
    test_loader = get_test_loader(GPU)

    # FP16 loader wrapper — inputs must also be half
    @torch.no_grad()
    def evaluate_fp16(model, loader):
        model.eval()
        preds, labels = [], []
        for x, y in loader:
            preds.extend(model(x.to(GPU).half()).argmax(1).cpu().numpy())
            labels.extend(y.numpy())
        p, l = np.array(preds), np.array(labels)
        return dict(
            accuracy  = float(accuracy_score(l, p)),
            precision = float(precision_score(l, p, average='macro', zero_division=0)),
            recall    = float(recall_score(l, p, average='macro', zero_division=0)),
            f1        = float(f1_score(l, p, average='macro', zero_division=0)),
        )

    metrics  = evaluate_fp16(model, test_loader)
    gpu_ms   = measure_gpu_ms(model.half(), GPU)
    size_mb  = disk_mb(model, use_state_dict=True)

    result = dict(
        track          = "C",
        method         = "GPU FP16 (.half())",
        device         = str(GPU),
        dtype          = "float16 (all layers)",
        **{k: round(v, 6) for k, v in metrics.items()},
        accuracy_drop  = round(baseline_acc - metrics['accuracy'], 6),
        size_mb        = size_mb,
        inference_cpu_ms = None,
        inference_gpu_ms = round(gpu_ms, 4) if gpu_ms else None,
        timing_note    = (
            "FP16 is not INT8 quantization but is a standard precision reduction "
            "technique. No calibration required. 2× memory vs FP32. "
            "GPU timing comparable to pruning benchmarks."
        ),
    )
    print(f"  acc={metrics['accuracy']:.4f}  drop={result['accuracy_drop']:+.4f}  "
          f"size={size_mb:.2f} MB  gpu={gpu_ms:.2f} ms")
    return result, model

# ══════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════
def main():
    print("\n" + "="*60)
    print("PTQ v2 — GPU + CPU Quantization")
    print("="*60)

    with open(BASELINE_JSON) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]

    # Baseline GPU timing (for fair comparison reference)
    print("\nMeasuring baseline GPU inference time...")
    fp32_gpu    = load_fp32(GPU)
    test_loader = get_test_loader(GPU)
    baseline_gpu_ms = measure_gpu_ms(fp32_gpu, GPU)
    baseline_cpu_ms = measure_cpu_ms(load_fp32(CPU))
    print(f"  Baseline GPU: {baseline_gpu_ms:.3f} ms  |  CPU: {baseline_cpu_ms:.3f} ms")

    result_a, model_a = run_track_a(baseline_acc)
    result_b, model_b = run_track_b(baseline_acc)
    result_c, model_c = run_track_c(baseline_acc)

    # ── Save models ───────────────────────────────────────────
    os.makedirs("__1__PTQ_v2_models", exist_ok=True)
    if model_a:
        torch.save(model_a, "__1__PTQ_v2_models/ptq_cpu_int8_histogram.pt")
        print("\nSaved: ptq_cpu_int8_histogram.pt")
    if model_b:
        torch.save(model_b.state_dict(), "__1__PTQ_v2_models/ptq_gpu_bnb_int8.pth")
        print("Saved: ptq_gpu_bnb_int8.pth")
    if model_c:
        torch.save(model_c.state_dict(), "__1__PTQ_v2_models/ptq_gpu_fp16.pth")
        print("Saved: ptq_gpu_fp16.pth")

    # ── Save JSON ─────────────────────────────────────────────
    output = {
        "_meta": {
            "method"     : "post_training_quantization_v2",
            "description": (
                "Three quantization tracks for fair cross-method comparison. "
                "Track A: CPU INT8 (torch.ao) — matches existing PTQ script. "
                "Track B: GPU INT8 (bitsandbytes) — GPU timing comparable to pruning. "
                "Track C: GPU FP16 (torch.half) — no library, 2x memory reduction."
            ),
            "comparison_guide": {
                "vs_pruning_inference_ms": (
                    "Use Track B or C gpu_ms values to compare with pruning. "
                    "Track A cpu_ms must NOT be compared with GPU pruning times."
                ),
                "vs_pruning_size_mb": (
                    "All tracks: size_mb is the real .pth file size on disk. "
                    "Directly comparable to pruning size_mb values."
                ),
                "fair_comparison_table": (
                    "Columns: method | device | dtype | acc | size_mb | inference_ms. "
                    "Only compare inference_ms within same device column."
                ),
            },
            "device_gpu"  : str(GPU),
            "bnb_version" : str(bnb.__version__) if BNB_AVAILABLE else "not installed",
        },
        "baseline": {
            **baseline,
            "inference_gpu_ms": round(baseline_gpu_ms, 4) if baseline_gpu_ms else None,
            "inference_cpu_ms": round(baseline_cpu_ms, 4),
        },
        "results": {
            "track_A_cpu_int8"  : result_a,
            "track_B_gpu_int8"  : result_b,
            "track_C_gpu_fp16"  : result_c,
        },
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(output, f, indent=2)

    # ── Summary ───────────────────────────────────────────────
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print(f"  {'Method':<30} {'Device':<6} {'Acc':>7} {'Drop':>7} {'MB':>7} {'GPU ms':>9} {'CPU ms':>9}")
    print(f"  {'-'*75}")
    print(f"  {'Baseline (FP32)':<30} {'gpu':<6} {baseline_acc:>7.4f} {'—':>7} "
          f"{baseline.get('size_mb', '—'):>7}  "
          f"{baseline_gpu_ms:>8.2f}  {baseline_cpu_ms:>8.2f}")
    for label, res in [("A: CPU INT8 (torch.ao)", result_a),
                        ("B: GPU INT8 (bitsandbytes)", result_b),
                        ("C: GPU FP16 (.half())", result_c)]:
        if res is None:
            print(f"  {label:<30} {'—':<6} {'SKIPPED':>7}")
            continue
        gpu = f"{res['inference_gpu_ms']:>8.2f}" if res['inference_gpu_ms'] else f"{'—':>8}"
        cpu = f"{res['inference_cpu_ms']:>8.2f}" if res['inference_cpu_ms'] else f"{'—':>8}"
        print(f"  {label:<30} {res['device']:<6} {res['accuracy']:>7.4f} "
              f"{res['accuracy_drop']:>+7.4f} {res['size_mb']:>7.2f} {gpu}  {cpu}")

    print(f"\n  Saved → {OUTPUT_JSON}")
    print(f"  Models → __1__PTQ_v2_models/")

if __name__ == "__main__":
    main()

GPU device : cuda
CUDA avail : True
bitsandbytes : NOT installed — Track B skipped
  Install: pip install bitsandbytes --break-system-packages

PTQ v2 — GPU + CPU Quantization

Measuring baseline GPU inference time...
  Baseline GPU: 5.401 ms  |  CPU: 17.931 ms

TRACK A — CPU PTQ (torch.ao, histogram observer)
  Device: CPU | INT8 weights + activations
  NOTE: CPU timing — not directly comparable to GPU pruning
  acc=0.9315  drop=+0.0005  size=24.11 MB  cpu=51.30 ms

TRACK B — SKIPPED (bitsandbytes not installed)

TRACK C — GPU FP16 (torch .half())
  Device: CUDA | 16-bit floats, no calibration needed
  timing directly comparable to GPU pruning results
  acc=0.9319  drop=+0.0001  size=47.26 MB  gpu=57.05 ms

Saved: ptq_cpu_int8_histogram.pt
Saved: ptq_gpu_fp16.pth

SUMMARY
  Method                         Device     Acc    Drop      MB    GPU ms    CPU ms
  ---------------------------------------------------------------------------
  Baseline (FP32)                gpu     0.9320       